# Debug Subspace GLP - Identity Mapping Validation

This notebook loads a SubspaceGLP model and verifies that the `project` -> `unproject` pipeline forms an identity mapping (A -> A) when flow matching is bypassed. It tests both a loaded model and a randomly initialized dummy model.

In [ ]:
import os, sys
from pathlib import Path

# ensure repo root on path
benchmark_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "Steering" / "post_process").exists():
        benchmark_root = candidate
        break
if benchmark_root is None:
    raise RuntimeError("Could not locate the Benchmark workspace root")
if str(benchmark_root) not in sys.path:
    sys.path.insert(0, str(benchmark_root))

import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Setup paths (modify if needed)
glp_source = "GLP/subspace-glp-uni-100M"
glp_checkpoint = "checkpoints/final"

print(f"Testing checkpoint: {glp_source} @ {glp_checkpoint}")

In [ ]:
from Steering.post_process.subspace import SubspaceGLP

loaded_sglp = None
dummy_sglp = None

# 1. Load SubspaceGLP
try:
    loaded_sglp = SubspaceGLP.load(glp_source, device=device, checkpoint=glp_checkpoint)
    print("Successfully loaded SubspaceGLP artifacts.")
    print(f"Subspace dimensions (k): {loaded_sglp.P.shape[1]}")
    print(f"Ambient dimensions (D): {loaded_sglp.P.shape[0]}")
except Exception as e:
    print(f"Failed to load SubspaceGLP artifacts. Error: {e}")

# 2. Create Dummy SubspaceGLP
class DummyGLP:
    pass

D = 2304
k = 256
P = torch.randn(D, k, device=device)
# Orthogonalize P to make it a valid projection matrix
U, _, _ = torch.svd(P)
P = U[:, :k]

mean_AB = torch.randn(D, device=device)
sub_mean = torch.randn(k, device=device)
sub_std = torch.rand(k, device=device) + 0.1
weights = torch.ones(k, device=device)

dummy_sglp = SubspaceGLP(DummyGLP(), P, mean_AB, sub_mean, sub_std, weights, device=device)
print("\nCreated Dummy SubspaceGLP.")


In [ ]:
def test_identity(sglp, name):
    print(f"\n{'='*50}")
    print(f"TESTING: {name}")
    print(f"{'='*50}")
    
    # Create a batch of random activations A
    N = 100
    D = sglp.P.shape[0]

    A = torch.randn(N, D, device=device) * 5.0 + 2.0
    print(f"Input A shape: {tuple(A.shape)}")

    # 1. Project A to subspace (normalizes internally)
    h_norm, h_null = sglp.project(A)

    print(f"h_norm shape: {tuple(h_norm.shape)} (Projected & Normalized)")
    print(f"h_null shape: {tuple(h_null.shape)} (Null space residual)")

    # 2. Unproject back to ambient space (denormalizes internally)
    A_out = sglp.unproject(h_norm, h_null)

    print(f"Output A_out shape: {tuple(A_out.shape)}")

    # 3. Validation
    diff = torch.norm(A - A_out, dim=-1)
    max_error = diff.max().item()
    mean_error = diff.mean().item()

    print("\n--- IDENTITY MAPPING TEST ---")
    print(f"Max L2 Error per vector: {max_error:.8e}")
    print(f"Mean L2 Error per vector: {mean_error:.8e}")

    if torch.allclose(A, A_out, atol=1e-4):
        print(f"\n✅ {name} PASSED: The project -> unproject pipeline is a perfect identity mapping.")
    else:
        print(f"\n❌ {name} FAILED: A != A_out. Check normalization / projection logic.")

if loaded_sglp is not None:
    test_identity(loaded_sglp, "Loaded SubspaceGLP")

if dummy_sglp is not None:
    test_identity(dummy_sglp, "Dummy SubspaceGLP")
